In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")


Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [ ]:
import numpy as np
import pandas as pd
import torch

def create_graph_snapshots(df, feature_cols, label_col='label', window_size=2000, overlap=1500):
    """
    Cắt DataFrame thành các Graph Snapshots dựa trên cửa sổ trượt (mỗi snapshot chứa window_size luồng, và có overlap luồng với snapshot trước đó).
    
    Args:
        df (pd.DataFrame): DataFrame đã được sort theo timestamp.
        feature_cols (list): Danh sách các cột đặc trưng hành vi làm node features.
        label_col (str): Tên cột chứa nhãn tấn công.
        window_size (int): Số lượng luồng (flow) trong một snapshot.
        overlap (int): Số lượng luồng giữ lại từ snapshot trước đó.
        
    Returns:
        list: Danh sách các dictionary, mỗi dict là một snapshot chứa 'x', 'y', và 'meta'.
    """
    snapshots = []
    
    # Tính bước trượt (stride)
    stride = window_size - overlap
    if stride <= 0:
        raise ValueError("Overlap phải nhỏ hơn window_size!")
        
    num_flows = len(df)
    

    X_all = df[feature_cols].values.astype(np.float32)
    Y_all = df[label_col].values.astype(np.int64)
    
    # Metadata cho việc nối cạnh
    dst_ips = df['dst_ip'].values
    dst_ports = df['dst_port'].values
    timestamps = df['timestamp_num'].values
    
    print(f"Bắt đầu tạo snapshot: Tổng số {num_flows} flows, Window={window_size}, Stride={stride}")
    
    # Quét qua toàn bộ dữ liệu với bước nhảy là stride
    for start_idx in range(0, num_flows - window_size + 1, stride):
        end_idx = start_idx + window_size
        
        # 1. Trích xuất Node Features và Labels cho Snapshot hiện tại
        x_snapshot = X_all[start_idx:end_idx]
        y_snapshot = Y_all[start_idx:end_idx]
        
        # 2. Trích xuất Metadata tương ứng
        meta_snapshot = {
            'dst_ip': dst_ips[start_idx:end_idx],
            'dst_port': dst_ports[start_idx:end_idx],
            'timestamp': timestamps[start_idx:end_idx],
            'global_indices': np.arange(start_idx, end_idx) # Lưu lại index gốc để track
        }
        
        # 3. Đóng gói vào dictionary
        snapshot_data = {
            'x': torch.tensor(x_snapshot),
            'y': torch.tensor(y_snapshot),
            'meta': meta_snapshot
        }
        
        snapshots.append(snapshot_data)
        
    print(f"Đã tạo thành công {len(snapshots)} snapshots!")
    return snapshots

In [ ]:
# tạo tham số cho cửa sổ trượt
WINDOW_SIZE = 2000
OVERLAP = 1500
feature_cols = [col for col in train_df.columns if col not in ["flow_id",'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'label', 'timestamp_num', "src_port", "dst_port"]]
print(f"Các cột đặc trưng được sử dụng: {len(feature_cols)} cột")
# Tạo snapshots cho tập Train
train_snapshots = create_graph_snapshots(
    df=train_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

# Tạo snapshots cho tập Valid và Test
valid_snapshots = create_graph_snapshots(
    df=valid_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

test_snapshots = create_graph_snapshots(
    df=test_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

Các cột đặc trưng được sử dụng: 314 cột
Bắt đầu tạo snapshot: Tổng số 2470638 flows, Window=2000, Stride=500
Đã tạo thành công 4938 snapshots!
Bắt đầu tạo snapshot: Tổng số 570134 flows, Window=2000, Stride=500
Đã tạo thành công 1137 snapshots!
Bắt đầu tạo snapshot: Tổng số 760240 flows, Window=2000, Stride=500
Đã tạo thành công 1517 snapshots!


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from tqdm import tqdm  # Thư viện tạo thanh tiến trình chuyên nghiệp

def build_graph_from_snapshot(snapshot, k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Nối cạnh cho một snapshot đơn lẻ
    Args:
        snapshot (dict): Dictionary chứa "x", "y và "meta"
        k (int): số lượng láng giềng tối đa có thể nối
        beta: hệ số trọng số cho Time Decay
        lambda_decay: hệ số điều chỉnh độ nhạy của Time Decay
    """
    x = snapshot['x']
    y = snapshot['y']
    meta = snapshot['meta']
    num_nodes = x.shape[0]

    target_ids = np.array([f"{ip}:{port}" for ip, port in zip(meta['dst_ip'], meta['dst_port'])])
    timestamps = meta['timestamp']
    edge_index_list = []

    # vòng lặp nối cạnh: nối cạnh nếu cùng dst_ip và dst_port, ưu tiên những node có timestamp gần nhau nhất
    # weight cạnh = alpha * cosine_similarity + beta * time_decay
    for i in range(num_nodes):
        same_target_mask = (target_ids == target_ids[i])
        same_target_indices = np.where(same_target_mask)[0]
        same_target_indices = same_target_indices[same_target_indices != i]

        if len(same_target_indices) == 0:
            continue

        time_diffs = np.abs(timestamps[same_target_indices] - timestamps[i])
        k_actual = min(k, len(time_diffs))
        top_k_local_indices = np.argsort(time_diffs)[:k_actual]
        top_k_global_indices = same_target_indices[top_k_local_indices]

        for j in top_k_global_indices:
            edge_index_list.append([i, j])

    if not edge_index_list:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

    # Tính Cosine Similarity
    src_nodes_features = x[edge_index[0]]
    dst_nodes_features = x[edge_index[1]]
    cos_sim = F.cosine_similarity(src_nodes_features, dst_nodes_features, dim=1, eps=1e-8)
    cos_sim = (cos_sim + 1.0) / 2.0 

    # Tính Time Decay với chuẩn hóa Min-Max cục bộ (để phù hợp với range của cosine similarity)
    src_times = torch.tensor(timestamps[edge_index[0].numpy()], dtype=torch.float32)
    dst_times = torch.tensor(timestamps[edge_index[1].numpy()], dtype=torch.float32)
    time_diffs_tensor = torch.abs(src_times - dst_times)
    
    max_diff = time_diffs_tensor.max()
    if max_diff > 0:
        time_diffs_norm = time_diffs_tensor / max_diff
    else:
        time_diffs_norm = time_diffs_tensor
        
    time_decay = torch.exp(-lambda_decay * time_diffs_norm)
    edge_weight = alpha * cos_sim + beta * time_decay

    return Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)


def convert_all_snapshots_to_graphs(snapshots_list, dataset_name="Train Set", k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Hàm wrapper bọc tqdm bên ngoài để track tiến trình tạo đồ thị cho danh sách snapshots.
    Args:
        snapshots_list (list): Danh sách các snapshot (mỗi snapshot là một dict chứa "x", "y", "meta")
        dataset_name (str): Tên của tập dữ liệu (dùng để hiển thị trên thanh tiến trình)
        k, alpha, beta, lambda_decay: Tham số cho hàm build_graph_from_snapshot
    """
    graph_objects = []
    
    # tqdm sẽ tự động tính toán % tiến độ, số snapshot/s và thời gian còn lại
    progress_bar = tqdm(
        snapshots_list, 
        desc=f"Building Graphs ({dataset_name})", 
        unit="snapshot",
        ncols=100 # Độ rộng của thanh tiến trình hiển thị trên console
    )
    
    for snap in progress_bar:
        graph_data = build_graph_from_snapshot(
            snapshot=snap, 
            k=k, 
            alpha=alpha, 
            beta=beta, 
            lambda_decay=lambda_decay
        )
        graph_objects.append(graph_data)
        
    return graph_objects


KeyboardInterrupt



In [ ]:
# tạo đồ thị cho các tập train, valid và test
train_graphs = convert_all_snapshots_to_graphs(train_snapshots, dataset_name="Train Set")


valid_graphs = convert_all_snapshots_to_graphs(valid_snapshots, dataset_name="Valid Set")


test_graphs = convert_all_snapshots_to_graphs(test_snapshots, dataset_name="Test Set")

Building Graphs (Test Set): 100%|█████████████████████████| 1517/1517 [04:07<00:00,  6.14snapshot/s]


In [ ]:
from torch_geometric.loader import DataLoader

# Thiết lập Batch Size (Số lượng snapshot đưa vào GPU trong 1 bước)
BATCH_SIZE = 32 

# Khởi tạo DataLoader cho cả 3 tập
# Tập Train cần shuffle=True để mô hình học khách quan, Valid/Test giữ nguyên thứ tự
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã chuẩn bị xong DataLoader:")
print(f" - Train Batches: {len(train_loader)}")
print(f" - Valid Batches: {len(valid_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Đã chuẩn bị xong DataLoader:
 - Train Batches: 155
 - Valid Batches: 36
 - Test Batches: 48


In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GCNConv

# code theo đúng cấu trúc trong paper DynamicIDS
class PaperBaselineGCN(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(PaperBaselineGCN, self).__init__()
        
        # 1. Input transformation layer (Nén số lượng feature đầu vào về 32)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. GCN Layers (Cấu trúc giống hệt paper: 32 -> 64 -> 128 -> 64)
        self.conv1 = GCNConv(32, 64)
        self.conv2 = GCNConv(64, 128)
        self.conv3 = GCNConv(128, 64)
        
        # 3. Output layer (Từ 64 chiều về 11 class)
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_weight):
        # --- Lớp đầu vào ---
        x = self.lin_in(x)
        x = F.relu(x) # Hàm kích hoạt phi tuyến sau phép biến đổi tuyến tính
        
        # --- Lớp GCN 1 ---
        # Truyền cả node features (x), cấu trúc cạnh (edge_index) và trọng số cạnh (edge_weight)
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training) # Dropout 0.5 như paper để chống Overfitting
        
        # --- Lớp GCN 2 ---
        x = self.conv2(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # --- Lớp GCN 3 ---
        x = self.conv3(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # --- Lớp Đầu ra ---
        x = self.lin_out(x)
        
        # Sử dụng log_softmax thay vì softmax thuần. 
        return F.log_softmax(x, dim=1)

In [ ]:
# Khởi tạo mô hình
model = PaperBaselineGCN(num_node_features=314, num_classes=11)

# chuyển mô hình lên GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Khai báo Optimizer (Adam là chuẩn mực nhất)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# 1. tạo class weights cho loss function
import sklearn.utils.class_weight as class_weight

unique_classes = np.unique(train_df['label']) # Lấy các class có trong tập train
all_train_labels = train_df['label'].values

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=all_train_labels
)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
class_weights = np.power(class_weights, 0.4) 
class_weights = np.clip(class_weights, a_min=0.5, a_max=6.0) 

# Đưa lên thiết bị (GPU/CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device) 

# ==========================================
# 2. Áp dụng vào Hàm Mất Mát (Loss Function)
# ==========================================
# Chèn class_weights_tensor vào tham số 'weight' của NLLLoss
criterion = torch.nn.NLLLoss(weight=class_weights_tensor)

print("Đã tích hợp Class Weights vào hàm Loss thành công!")
print(f"Trọng số các class: {class_weights}")

print(model)

Đã tích hợp Class Weights vào hàm Loss thành công!
Trọng số các class: [1.64730877 0.5        3.7653502  1.29446194 1.67705869 3.80442181
 2.0250111  0.83822433 1.76304403 1.22605595 1.69183888]
PaperBaselineGCN(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (conv1): GCNConv(32, 64)
  (conv2): GCNConv(64, 128)
  (conv3): GCNConv(128, 64)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)


In [ ]:
import torch
import numpy as np

# class EarlyStopping để theo dõi macro f1 trên tập validation
class EarlyStopping:
    """
    Class EarlyStopping theo macro f1 trên tập validation
    """
    def __init__(self, patience=8, path='best_baseline_gcn.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    # hàm call: mỗi khi object(val_f1, model) được gọi thì sẽ chạy hàm này
    def __call__(self, val_f1, model):
        # Nếu F1 cải thiện (lớn hơn điểm tốt nhất trước đó)
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.save_checkpoint(model)
            self.counter = 0 # Reset bộ đếm về 0
        else:
            # Nếu không cải thiện, tăng bộ đếm thêm 1
            self.counter += 1
            print(f' -> [EarlyStopping] Bộ đếm tăng: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        '''Lưu mô hình khi điểm Validation F1 tăng'''
        torch.save(model.state_dict(), self.path)
        print(f' -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.')

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score


def train_epoch(epoch, EPOCHS):
    """Hàm huấn luyện cho một epoch

    Args:
        epoch (int): số epoch hiện tại
        EPOCHS (int): số epoch tổng cộng để huấn luyện (dùng để hiển thị trên thanh tiến trình)

    Returns:
        loss_avg (float): loss trung bình trên tập train
        macro_f1 (float): điểm Macro F1 trên tập train
    """
    
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Train
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Cập nhật số loss liên tục trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    loss_avg = total_loss / len(train_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

@torch.no_grad()
def valid_epoch(epoch, EPOCHS):
    """Hàm đánh giá trên tập validation

    Args:
        epoch (int): số epoch hiện tại
        EPOCHS (int): số epoch tổng cộng để huấn luyện (dùng để hiển thị trên thanh tiến trình)

    Returns:
        loss_avg (float): loss trung bình trên tập validation
        macro_f1 (float): điểm Macro F1 trên tập validation
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Valid
    progress_bar = tqdm(valid_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Valid]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
    loss_avg = total_loss / len(valid_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

In [ ]:
EPOCHS = 50

# early_stoppping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # chạy 1 epoch
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # kiểm tra early stopping
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break

Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.9928 - Macro F1: 0.3698 | Valid Loss: 0.3528 - Macro F1: 0.5901
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.5198 - Macro F1: 0.6229 | Valid Loss: 0.3108 - Macro F1: 0.7648
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.3622 - Macro F1: 0.7331 | Valid Loss: 0.2274 - Macro F1: 0.7975
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.2484 - Macro F1: 0.8123 | Valid Loss: 0.2346 - Macro F1: 0.8081
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 046/50 | Train Loss: 0.0404 - Macro F1: 0.9638 | Valid Loss: 0.1152 - Macro F1: 0.9317
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


Epoch: 047/50 | Train Loss: 0.0389 - Macro F1: 0.9679 | Valid Loss: 0.1131 - Macro F1: 0.9392
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 048/50 | Train Loss: 0.0363 - Macro F1: 0.9707 | Valid Loss: 0.1499 - Macro F1: 0.9231
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


Epoch: 049/50 | Train Loss: 0.1065 - Macro F1: 0.9440 | Valid Loss: 0.1099 - Macro F1: 0.9285
 -> [EarlyStopping] Bộ đếm tăng: 2 / 8


Epoch: 050/50 | Train Loss: 0.0436 - Macro F1: 0.9639 | Valid Loss: 0.0792 - Macro F1: 0.9407
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


In [ ]:
# chạy thêm 50 epoch nữa
EXTRA_EPOCHS = 50
CURRENT_EPOCH = 50 # số epoch đã chạy được

print(f"Tiếp tục huấn luyện từ Epoch {CURRENT_EPOCH + 1}...")

for epoch in range(CURRENT_EPOCH + 1, CURRENT_EPOCH + EXTRA_EPOCHS + 1):
    # 1. Chạy huấn luyện
    train_loss, train_f1 = train_epoch(epoch, CURRENT_EPOCH + EXTRA_EPOCHS)
    
    # 2. Chạy validation
    valid_loss, valid_f1 = valid_epoch(epoch, CURRENT_EPOCH + EXTRA_EPOCHS)
    
    print(f"Epoch: {epoch:03d}/{CURRENT_EPOCH + EXTRA_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Tiếp tục check Early Stopping (Nó sẽ nhớ best_f1 hiện tại là 0.9407)
    early_stopping(valid_f1, model)
    
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Kích hoạt tại Epoch {epoch}! Đỉnh cao cuối cùng đã được chốt.")
        break

Tiếp tục huấn luyện từ Epoch 51...


Epoch: 051/100 | Train Loss: 0.0456 - Macro F1: 0.9659 | Valid Loss: 0.0900 - Macro F1: 0.9300
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


Epoch: 052/100 | Train Loss: 0.0344 - Macro F1: 0.9746 | Valid Loss: 0.0902 - Macro F1: 0.9379
 -> [EarlyStopping] Bộ đếm tăng: 2 / 8


Epoch: 053/100 | Train Loss: 0.0481 - Macro F1: 0.9638 | Valid Loss: 0.0826 - Macro F1: 0.9364
 -> [EarlyStopping] Bộ đếm tăng: 3 / 8


Epoch: 054/100 | Train Loss: 0.0370 - Macro F1: 0.9697 | Valid Loss: 0.0928 - Macro F1: 0.9365
 -> [EarlyStopping] Bộ đếm tăng: 4 / 8


Epoch: 055/100 | Train Loss: 0.0498 - Macro F1: 0.9616 | Valid Loss: 0.1253 - Macro F1: 0.9332
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 056/100 | Train Loss: 0.0555 - Macro F1: 0.9677 | Valid Loss: 0.1134 - Macro F1: 0.9334
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 057/100 | Train Loss: 0.0427 - Macro F1: 0.9703 | Valid Loss: 0.1600 - Macro F1: 0.9275
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 058/100 | Train Loss: 0.0409 - Macro F1: 0.9639 | Valid Loss: 0.0970 - Macro F1: 0.9372
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Kích hoạt tại Epoch 58! Đỉnh cao cuối cùng đã được chốt.


In [ ]:
from sklearn.metrics import f1_score
from tqdm import tqdm
import torch

@torch.no_grad()
def test_benchmark_new_nodes(loader, window_size=2000, stride=500):
    """Hàm đánh giá trên tập Test với chiến lược chỉ đánh giá luồng mới (bỏ qua phần overlap)
    Args:
        loader: DataLoader cho tập Test
        window_size: kích thước cửa sổ (số node trong mỗi snapshot)
        stride: bước nhảy giữa các snapshot (window_size - overlap)
    Returns:
        final_macro_f1 (float): điểm Macro F1 cuối cùng trên tập Test
        all_preds (list): danh sách dự đoán của mô hình cho tất cả node được đánh giá
        all_labels (list): danh sách nhãn thực tế tương ứng với các node
    """
    model.eval() # Tắt tính toán gradient, tắt Dropout
    
    all_preds = []
    all_labels = []
    
    overlap = window_size - stride
    curr_snap_idx = 0
    
    print("Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...")
    progress_bar = tqdm(loader, desc="Test Benchmark", ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        
        # Chạy forward pass qua mô hình
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # Dự đoán class có xác suất cao nhất ngay lập tức
        preds = out.argmax(dim=1)
        
        # Bóc tách từng đồ thị trong batch để cắt lấy các Node mới
        for g_idx in range(batch.num_graphs):
            # Tạo mask để lấy đúng các node thuộc về đồ thị thứ g_idx
            node_mask = (batch.batch == g_idx)
            
            g_preds = preds[node_mask]
            g_labels = batch.y[node_mask]
            g_num_nodes = g_preds.size(0)
            
            # Logic cắt node mới
            if curr_snap_idx == 0:
                # Snapshot đầu tiên: Đánh giá toàn bộ
                start_eval_idx = 0
            else:
                # Các Snapshot sau: Bỏ qua phần overlap, chỉ lấy phần mới
                start_eval_idx = overlap
                
            # Đề phòng trường hợp snapshot cuối cùng bị thiếu node
            if start_eval_idx < g_num_nodes:
                new_preds = g_preds[start_eval_idx:g_num_nodes]
                new_labels = g_labels[start_eval_idx:g_num_nodes]
                
                # Nạp vào danh sách tổng
                all_preds.extend(new_preds.cpu().numpy())
                all_labels.extend(new_labels.cpu().numpy())
            
            curr_snap_idx += 1
            
    # Tính F1 Score cuối cùng
    final_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"Tổng số flow thực tế được đánh giá: {len(all_labels)}")
    print(f"=====================================")
    print(f"🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: {final_macro_f1:.4f}")
    print(f"=====================================")

    # in ra classification report chi tiết
    from sklearn.metrics import classification_report
    print("\n📊 Classification Report Chi Tiết:")
    print(classification_report(all_labels, all_preds, digits=4))
    
    return final_macro_f1, all_preds, all_labels

In [ ]:
# load lại trọng số tối ưu
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# chạy benchmark trên tập Test
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

Thử bằng model GAT

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv

# Thay các lớp GCNConv bằng GATConv, đồng thời thêm tham số edge_attr để tận dụng trọng số cạnh đã tính toán
class AdvancedGAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(AdvancedGAT, self).__init__()
        
        # 1. Lớp đầu vào (Giữ nguyên để nén feature)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GAT với Multi-head Attention
        # Lớp 1: Input 32 -> 4 heads x 16 = 64 chiều (Tương đương GCNConv 64)
        self.gat1 = GATConv(
            in_channels=32, out_channels=16, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 2: Input 64 -> 4 heads x 32 = 128 chiều (Tương đương GCNConv 128)
        self.gat2 = GATConv(
            in_channels=64, out_channels=32, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 3: Input 128 -> 4 heads x 64. 
        # CHÚ Ý: Ở lớp GNN cuối cùng, ta dùng concat=False để tính trung bình các heads, 
        # gom lại thành 64 chiều chuẩn bị cho lớp Linear.
        self.gat3 = GATConv(
            in_channels=128, out_channels=64, heads=4, concat=False, edge_dim=1
        )
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # GATConv yêu cầu edge_attr phải là ma trận 2 chiều [Số cạnh, Số feature cạnh].
        # Trọng số Cosine + Time Decay của ta đang là mảng 1 chiều [E], ta cần thêm 1 trục (unsqueeze).
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
            
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp GAT 1 (activation là elu thay vì relu để phù hợp với paper GAT gốc)
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 2
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 3
        x = self.gat3(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [ ]:
# 1. Khởi tạo GAT Model
model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)



# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)


In [ ]:
EPOCHS = 50


early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break

Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.8365 - Macro F1: 0.4541 | Valid Loss: 0.2868 - Macro F1: 0.6832
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3433 - Macro F1: 0.7353 | Valid Loss: 0.2337 - Macro F1: 0.7143
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1634 - Macro F1: 0.8717 | Valid Loss: 0.1314 - Macro F1: 0.8963
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.0901 - Macro F1: 0.9231 | Valid Loss: 0.1193 - Macro F1: 0.8864
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


...

Epoch: 046/50 | Train Loss: 0.0301 - Macro F1: 0.9698 | Valid Loss: 0.0952 - Macro F1: 0.9275
 -> [EarlyStopping] Bộ đếm tăng: 4 / 8


Epoch: 047/50 | Train Loss: 0.0300 - Macro F1: 0.9752 | Valid Loss: 0.1162 - Macro F1: 0.9294
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 048/50 | Train Loss: 0.0308 - Macro F1: 0.9761 | Valid Loss: 0.1108 - Macro F1: 0.9284
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 049/50 | Train Loss: 0.0543 - Macro F1: 0.9694 | Valid Loss: 0.0801 - Macro F1: 0.9334
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 050/50 | Train Loss: 0.0244 - Macro F1: 0.9788 | Valid Loss: 0.0692 - Macro F1: 0.9378
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 50!


In [79]:
# 1. Nạp lại trọng số tối ưu nhất đã được Early Stopping lưu lại trước đó
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13020\770146532.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark:   0%|                                                        | 0/48 [00:00<?, ?it/s]

Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:06<00:00,  7.99it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8612

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9389    0.8649    0.9004     19846
           1     0.9998    1.0000    0.9999    484077
           2     0.5460    0.9932    0.7047      2515
           3     0.9982    0.8803    0.9355     36253
           4     0.6361    0.8132    0.7138     18979
           5     0.9980    0.9996    0.9988      2451
           6     0.5059    0.7453    0.6027     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.7201    0.9935    0.8350     16746
           9     0.9998    0.6721    0.8038     41523
          10     0.9665    0.9968    0.9814     18567

    accuracy                         0.9633    760000
   macro avg     0.8463    0.9049    0.8612    760000
weighted avg     0.9729    0.9633    0.9648    760000



In [86]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.8722 - Macro F1: 0.4409 | Valid Loss: 0.3198 - Macro F1: 0.7122
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3722 - Macro F1: 0.7357 | Valid Loss: 0.2283 - Macro F1: 0.7885
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1925 - Macro F1: 0.8558 | Valid Loss: 0.1214 - Macro F1: 0.8892
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1308 - Macro F1: 0.9068 | Valid Loss: 0.1170 - Macro F1: 0.9087
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 033/50 | Train Loss: 0.0246 - Macro F1: 0.9798 | Valid Loss: 0.0640 - Macro F1: 0.9400
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_13020\1205045762.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Epoch: 034/50 | Train Loss: 0.0663 - Macro F1: 0.9516 | Valid Loss: 0.0938 - Macro F1: 0.9284
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 34!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:05<00:00,  8.85it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8741

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9239    0.8648    0.8934     19846
           1     0.9993    1.0000    0.9997    484077
           2     0.3562    0.9940    0.5244      2515
           3     0.9958    0.8728    0.9302     36253
           4     0.7448    0.7861    0.7649     18979
           5     0.9812    0.9996    0.9903      2451
           6     0.9148    0.6998    0.7930     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.6762    0.9983    0.8063     16746
           9     0.9997    0.8718    0.9314     41523
          10     0.9714    0.9965    0.9838     18567

    accuracy                         0.9725    760000
   macro avg     0.8694    0.9162    0.8741    760000
weighted avg     0.9797    0.9725    0.9742    760000



In [ ]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.9015 - Macro F1: 0.4171 | Valid Loss: 0.3507 - Macro F1: 0.5198
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3774 - Macro F1: 0.7250 | Valid Loss: 0.1847 - Macro F1: 0.8522
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1827 - Macro F1: 0.8759 | Valid Loss: 0.1534 - Macro F1: 0.8821
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1030 - Macro F1: 0.9200 | Valid Loss: 0.1236 - Macro F1: 0.9057
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 027/50 | Train Loss: 0.0296 - Macro F1: 0.9762 | Valid Loss: 0.0983 - Macro F1: 0.9252
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_13020\1205045762.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Epoch: 028/50 | Train Loss: 0.0296 - Macro F1: 0.9756 | Valid Loss: 0.0888 - Macro F1: 0.9265
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 28!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:03<00:00, 13.92it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8382

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.8940    0.8444    0.8685     19846
           1     0.9986    1.0000    0.9993    484077
           2     0.3673    0.9817    0.5346      2515
           3     0.9981    0.8528    0.9197     36253
           4     0.5990    0.8260    0.6944     18979
           5     0.9816    0.9996    0.9905      2451
           6     0.5083    0.7274    0.5984     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.7474    0.9018    0.8174     16746
           9     1.0000    0.6701    0.8025     41523
          10     0.9998    0.9958    0.9978     18567

    accuracy                         0.9593    760000
   macro avg     0.8267    0.8904    0.8382    760000
weighted avg     0.9709    0.9593    0.9616    760000



In [88]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.9322 - Macro F1: 0.3882 | Valid Loss: 0.3918 - Macro F1: 0.6098
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.4287 - Macro F1: 0.7043 | Valid Loss: 0.3111 - Macro F1: 0.7689
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.2501 - Macro F1: 0.8339 | Valid Loss: 0.1407 - Macro F1: 0.8880
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1282 - Macro F1: 0.9060 | Valid Loss: 0.1389 - Macro F1: 0.9065
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 045/50 | Train Loss: 0.0200 - Macro F1: 0.9846 | Valid Loss: 0.0581 - Macro F1: 0.9425
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_13020\1205045762.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Epoch: 046/50 | Train Loss: 0.0197 - Macro F1: 0.9825 | Valid Loss: 0.0724 - Macro F1: 0.9411
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 46!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:03<00:00, 13.81it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8747

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.8304    0.8975    0.8627     19846
           1     0.9993    1.0000    0.9997    484077
           2     0.3643    0.9813    0.5313      2515
           3     0.9990    0.8688    0.9294     36253
           4     0.7088    0.8141    0.7578     18979
           5     0.9855    0.9976    0.9915      2451
           6     0.8424    0.7399    0.7878     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.7865    0.9681    0.8679     16746
           9     0.9999    0.8386    0.9122     41523
          10     0.9711    0.9963    0.9835     18567

    accuracy                         0.9720    760000
   macro avg     0.8625    0.9179    0.8747    760000
weighted avg     0.9778    0.9720    0.9734    760000



GraphSAGE

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GraphConv

# thay các lớp GATConv bằng GraphConv, đồng thời thêm tham số edge_attr để tận dụng trọng số cạnh đã tính toán
class AdvancedGraphSAGE(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(AdvancedGraphSAGE, self).__init__()
        
        # 1. Lớp đầu vào (Nén đặc trưng thô xuống 32 chiều)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GraphConv (Đóng vai trò như GraphSAGE)
        # aggr='mean': Tổng hợp hàng xóm bằng trung bình cộng
        # Thuật toán sẽ học 2 tập trọng số riêng: W1 cho bản thân node, W2 cho hàng xóm
        self.sage1 = GraphConv(in_channels=32, out_channels=64, aggr='mean')
        self.sage2 = GraphConv(in_channels=64, out_channels=128, aggr='mean')
        self.sage3 = GraphConv(in_channels=128, out_channels=64, aggr='mean')
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # Nén Input
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp SAGE 1 (Sử dụng edge_attr để mồi thêm trọng số cạnh)
        x = self.sage1(x, edge_index, edge_weight=edge_attr)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp SAGE 2
        x = self.sage2(x, edge_index, edge_weight=edge_attr)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp SAGE 3
        x = self.sage3(x, edge_index, edge_weight=edge_attr)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [71]:
# 1. Khởi tạo mô hình GraphSAGE (Đảm bảo num_node_features = 314)
model = AdvancedGraphSAGE(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Reset Optimizer (Để nó bắt đầu theo dõi các tham số mới của GraphSAGE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# 3. Reset Early Stopping với tên file mới để không đè lên GAT/GCN
early_stopping = EarlyStopping(patience=8, path='best_advanced_sage.pt')

print("Đã nạp kiến trúc GraphSAGE! Sẵn sàng huấn luyện.")
print(model)

Đã nạp kiến trúc GraphSAGE! Sẵn sàng huấn luyện.
AdvancedGraphSAGE(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (sage1): GraphConv(32, 64)
  (sage2): GraphConv(64, 128)
  (sage3): GraphConv(128, 64)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)


In [72]:
EPOCHS = 50

print("Bắt đầu huấn luyện mô hình GraphSAGE...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    early_stopping(valid_f1, model)
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Kích hoạt tại Epoch {epoch}!")
        break

Bắt đầu huấn luyện mô hình GraphSAGE...


Epoch: 001/50 | Train Loss: 1.0850 - Macro F1: 0.2690 | Valid Loss: 0.4089 - Macro F1: 0.3394
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.5636 - Macro F1: 0.5331 | Valid Loss: 0.3589 - Macro F1: 0.5463
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.3975 - Macro F1: 0.6740 | Valid Loss: 0.2858 - Macro F1: 0.6908
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.3062 - Macro F1: 0.7421 | Valid Loss: 0.2813 - Macro F1: 0.7425
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 046/50 | Train Loss: 0.0581 - Macro F1: 0.9602 | Valid Loss: 0.0844 - Macro F1: 0.9102
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 047/50 | Train Loss: 0.0529 - Macro F1: 0.9592 | Valid Loss: 0.0820 - Macro F1: 0.9378
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 048/50 | Train Loss: 0.0425 - Macro F1: 0.9654 | Valid Loss: 0.0982 - Macro F1: 0.9241
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


Epoch: 049/50 | Train Loss: 0.0517 - Macro F1: 0.9502 | Valid Loss: 0.1068 - Macro F1: 0.9260
 -> [EarlyStopping] Bộ đếm tăng: 2 / 8


Epoch: 050/50 | Train Loss: 0.0349 - Macro F1: 0.9692 | Valid Loss: 0.0999 - Macro F1: 0.9109
 -> [EarlyStopping] Bộ đếm tăng: 3 / 8


In [74]:
EXTRA_EPOCHS = 50
CURRENT_EPOCH = 50 # Số epoch bạn đã chạy xong

print(f"Tiếp tục huấn luyện từ Epoch {CURRENT_EPOCH + 1}...")

for epoch in range(CURRENT_EPOCH + 1, CURRENT_EPOCH + EXTRA_EPOCHS + 1):
    # 1. Chạy huấn luyện
    train_loss, train_f1 = train_epoch(epoch, CURRENT_EPOCH + EXTRA_EPOCHS)
    
    # 2. Chạy validation
    valid_loss, valid_f1 = valid_epoch(epoch, CURRENT_EPOCH + EXTRA_EPOCHS)
    
    print(f"Epoch: {epoch:03d}/{CURRENT_EPOCH + EXTRA_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Tiếp tục check Early Stopping (Nó sẽ nhớ best_f1 hiện tại là 0.9407)
    early_stopping(valid_f1, model)
    
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Kích hoạt tại Epoch {epoch}! Đỉnh cao cuối cùng đã được chốt.")

Tiếp tục huấn luyện từ Epoch 51...


Epoch: 051/100 | Train Loss: 0.0546 - Macro F1: 0.9499 | Valid Loss: 0.0915 - Macro F1: 0.9375
 -> [EarlyStopping] Bộ đếm tăng: 4 / 8


Epoch: 052/100 | Train Loss: 0.0434 - Macro F1: 0.9648 | Valid Loss: 0.0823 - Macro F1: 0.9321
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 053/100 | Train Loss: 0.0352 - Macro F1: 0.9721 | Valid Loss: 0.0964 - Macro F1: 0.9350
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 054/100 | Train Loss: 0.0384 - Macro F1: 0.9704 | Valid Loss: 0.1108 - Macro F1: 0.9260
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 055/100 | Train Loss: 0.0480 - Macro F1: 0.9628 | Valid Loss: 0.0797 - Macro F1: 0.9140
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Kích hoạt tại Epoch 55! Đỉnh cao cuối cùng đã được chốt.


Epoch: 056/100 | Train Loss: 0.1161 - Macro F1: 0.9199 | Valid Loss: 0.1127 - Macro F1: 0.9035
 -> [EarlyStopping] Bộ đếm tăng: 9 / 8

🛑 [DỪNG SỚM] Kích hoạt tại Epoch 56! Đỉnh cao cuối cùng đã được chốt.


KeyboardInterrupt: 

In [75]:
model.load_state_dict(torch.load('best_advanced_sage.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13020\655051312.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_advanced_sage.pt'))


Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:02<00:00, 20.73it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8463

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9122    0.8084    0.8572     19846
           1     0.9994    1.0000    0.9997    484077
           2     0.3511    0.9952    0.5191      2515
           3     0.9974    0.8475    0.9164     36253
           4     0.5632    0.7521    0.6441     18979
           5     0.9330    0.9996    0.9651      2451
           6     0.8597    0.7182    0.7826     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.6659    0.9976    0.7987     16746
           9     0.9999    0.7493    0.8567     41523
          10     0.9514    0.9942    0.9723     18567

    accuracy                         0.9625    760000
   macro avg     0.8394    0.8961    0.8463    760000
weighted avg     0.9732    0.9625    0.9648    760000



Thử xây dựng với thời gian thay đổi đồ thị theo s

In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")

Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [2]:
def check_time_window_numeric(df, delta_t_seconds):
    """
    delta_t_seconds: Khoảng thời gian muốn thử (tính bằng giây).
    Ví dụ: 0.01 là 10 mili-giây, 0.1 là 100 mili-giây.
    """
    # Gom nhóm bằng cách chia lấy phần nguyên
    # Ép kiểu int để loại bỏ phần dư, biến nó thành 1 ID duy nhất cho mỗi cửa sổ
    window_ids = (df['timestamp_num'] / delta_t_seconds).astype(int)
    
    # Đếm số lượng flow trong mỗi ID (chính là 1 snapshot)
    flow_counts = window_ids.value_counts()
    
    print(f"--- Đánh giá cửa sổ Delta t = {delta_t_seconds} giây ---")
    print(f"Trung bình số flow / snapshot: {flow_counts.mean():.0f}")
    print(f"Lớn nhất (Đỉnh tấn công): {flow_counts.max()} flows")
    print(f"Nhỏ nhất (Lúc rảnh): {flow_counts.min()} flows")
    print(f"Tổng số snapshot sẽ tạo ra: {len(flow_counts)}")
    print("-" * 40)

# Chạy thử với 3 mốc mili-giây phổ biến:
check_time_window_numeric(train_df, 0.01) # Cửa sổ 10 ms (0.01s)
check_time_window_numeric(train_df, 0.05) # Cửa sổ 50 ms (0.05s)
check_time_window_numeric(train_df, 0.10) # Cửa sổ 100 ms (0.10s)

--- Đánh giá cửa sổ Delta t = 0.01 giây ---
Trung bình số flow / snapshot: 6
Lớn nhất (Đỉnh tấn công): 2670 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 403914
----------------------------------------
--- Đánh giá cửa sổ Delta t = 0.05 giây ---
Trung bình số flow / snapshot: 26
Lớn nhất (Đỉnh tấn công): 5835 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 96791
----------------------------------------
--- Đánh giá cửa sổ Delta t = 0.1 giây ---
Trung bình số flow / snapshot: 50
Lớn nhất (Đỉnh tấn công): 6796 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 49877
----------------------------------------


In [4]:
import pandas as pd
import numpy as np

def create_graph_snapshots_time_based(df, feature_cols, label_col='label', delta_t=0.1):
    """
    Tạo các snapshot đồ thị dựa trên cửa sổ thời gian tĩnh (Delta t).
    
    delta_t: Kích thước cửa sổ (tính bằng giây). Mặc định 0.1 = 100 mili-giây.
    """
    print(f"Đang băm dữ liệu theo cửa sổ thời gian Delta t = {delta_t} giây...")
    
    
    # 2. Tạo ID cho từng cửa sổ (Chia lấy phần nguyên)
    df['window_id'] = (df['timestamp_num'] / delta_t).astype(int)
    
    snapshots = []
    
    # 3. Gom nhóm (Groupby) các flow có chung Window ID lại thành 1 snapshot
    grouped = df.groupby('window_id')
    
    for window_id, group in grouped:
        x_snapshot = group[feature_cols].values
        y_snapshot = group[label_col].values
        
        meta_snapshot = {
            'dst_ip': group['dst_ip'].values,
            'dst_port': group['dst_port'].values,
            'timestamp': group['timestamp_num'].values,
            'global_indices': group.index.values 
        }
        
        snapshots.append({
            'x': x_snapshot,
            'y': y_snapshot,
            'meta': meta_snapshot
        })
        
    print(f"Hoàn tất! Đã tạo thành công {len(snapshots)} snapshots.")
    return snapshots

# ==========================================
# THỰC THI CHO TẬP TRAIN, VALID VÀ TEST
# ==========================================
# (Chạy lần lượt cho cả 3 tập để đồng bộ dữ liệu)
feature_cols = [col for col in train_df.columns if col not in ["flow_id",'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'label', 'timestamp_num', "src_port", "dst_port"]]
train_snapshots_time = create_graph_snapshots_time_based(train_df, feature_cols, label_col='label', delta_t=0.1)
valid_snapshots_time = create_graph_snapshots_time_based(valid_df, feature_cols, label_col='label', delta_t=0.1)
test_snapshots_time = create_graph_snapshots_time_based(test_df, feature_cols, label_col='label', delta_t=0.1)

Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 49877 snapshots.
Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 9369 snapshots.
Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 13313 snapshots.


In [7]:
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from tqdm import tqdm
def build_graph_from_snapshot(snapshot, k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Xử lý một snapshot đơn lẻ (Đã tối ưu CPU và chuẩn hóa cho Time-based).
    """
    # 1. ÉP KIỂU TENSOR NGAY TỪ ĐẦU (Rất quan trọng cho PyTorch Geometric)
    x = torch.tensor(snapshot['x'], dtype=torch.float32)
    y = torch.tensor(snapshot['y'], dtype=torch.long)
    meta = snapshot['meta']
    num_nodes = x.shape[0]

    # 2. CẮT SỚM (EARLY EXIT): Nếu chỉ có 1 node, trả về luôn đồ thị không cạnh
    # Giúp tiết kiệm hàng triệu chu kỳ CPU cho các snapshot lúc mạng rảnh rỗi
    if num_nodes <= 1:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    target_ids = np.array([f"{ip}:{port}" for ip, port in zip(meta['dst_ip'], meta['dst_port'])])
    timestamps = meta['timestamp']
    edge_index_list = []

    # 3. Vòng lặp nối cạnh Temporal K-NN (Logic của bạn rất chuẩn, giữ nguyên)
    for i in range(num_nodes):
        same_target_mask = (target_ids == target_ids[i])
        same_target_indices = np.where(same_target_mask)[0]
        same_target_indices = same_target_indices[same_target_indices != i]

        if len(same_target_indices) == 0:
            continue

        time_diffs = np.abs(timestamps[same_target_indices] - timestamps[i])
        # k_actual sẽ tự động bằng độ dài mảng nếu số node ít hơn k
        k_actual = min(k, len(time_diffs))
        top_k_local_indices = np.argsort(time_diffs)[:k_actual]
        top_k_global_indices = same_target_indices[top_k_local_indices]

        for j in top_k_global_indices:
            edge_index_list.append([i, j])

    # 4. Rào lỗi số 2: Có nhiều node nhưng không có node nào chung IP/Port
    if not edge_index_list:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

    # 5. Tính Cosine Similarity (x lúc này đã là Torch Tensor nên tính toán an toàn)
    src_nodes_features = x[edge_index[0]]
    dst_nodes_features = x[edge_index[1]]
    cos_sim = F.cosine_similarity(src_nodes_features, dst_nodes_features, dim=1, eps=1e-8)
    cos_sim = (cos_sim + 1.0) / 2.0 

    # 6. Tính Time Decay với chuẩn hóa Min-Max cục bộ
    src_times = torch.tensor(timestamps[edge_index[0].numpy()], dtype=torch.float32)
    dst_times = torch.tensor(timestamps[edge_index[1].numpy()], dtype=torch.float32)
    time_diffs_tensor = torch.abs(src_times - dst_times)
    
    max_diff = time_diffs_tensor.max()
    if max_diff > 0:
        time_diffs_norm = time_diffs_tensor / max_diff
    else:
        time_diffs_norm = time_diffs_tensor
        
    time_decay = torch.exp(-lambda_decay * time_diffs_norm)
    edge_weight = alpha * cos_sim + beta * time_decay

    return Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)


def convert_all_snapshots_to_graphs(snapshots_list, dataset_name="Train Set", k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Hàm wrapper bọc tqdm bên ngoài để track tiến trình tạo đồ thị cho TOÀN BỘ danh sách snapshots.
    """
    graph_objects = []
    
    # tqdm sẽ tự động tính toán % tiến độ, số snapshot/s và thời gian còn lại
    progress_bar = tqdm(
        snapshots_list, 
        desc=f"Building Graphs ({dataset_name})", 
        unit="snapshot",
        ncols=100 # Độ rộng của thanh tiến trình hiển thị trên console
    )
    
    for snap in progress_bar:
        graph_data = build_graph_from_snapshot(
            snapshot=snap, 
            k=k, 
            alpha=alpha, 
            beta=beta, 
            lambda_decay=lambda_decay
        )
        graph_objects.append(graph_data)
        
    return graph_objects

In [8]:
# Thực hiện tạo đồ thị và track tiến trình cho tập Train
train_graphs = convert_all_snapshots_to_graphs(train_snapshots_time, dataset_name="Train Set")

# Thực hiện tạo đồ thị và track tiến trình cho tập Valid
valid_graphs = convert_all_snapshots_to_graphs(valid_snapshots_time, dataset_name="Valid Set")

# Thực hiện tạo đồ thị và track tiến trình cho tập Test
test_graphs = convert_all_snapshots_to_graphs(test_snapshots_time, dataset_name="Test Set")

Building Graphs (Test Set): 100%|██████████████████████| 13313/13313 [00:18<00:00, 709.08snapshot/s]


In [9]:
from torch_geometric.loader import DataLoader

# Thiết lập Batch Size (Số lượng snapshot đưa vào GPU trong 1 bước)
# Với máy cấu hình thông thường, bạn có thể để 32 hoặc 64. 
BATCH_SIZE = 32 

# Khởi tạo DataLoader cho cả 3 tập
# Lưu ý: Tập Train cần shuffle=True để mô hình học khách quan, Valid/Test giữ nguyên thứ tự
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã chuẩn bị xong DataLoader:")
print(f" - Train Batches: {len(train_loader)}")
print(f" - Valid Batches: {len(valid_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Đã chuẩn bị xong DataLoader:
 - Train Batches: 1559
 - Valid Batches: 293
 - Test Batches: 417


In [10]:
import torch
import numpy as np

class EarlyStopping:
    """
    Trình quản lý dừng sớm (Early Stopping) để theo dõi Macro F1 trên tập Validation.
    """
    def __init__(self, patience=8, path='best_baseline_gcn.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    def __call__(self, val_f1, model):
        # Nếu F1 cải thiện (lớn hơn điểm tốt nhất trước đó)
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.save_checkpoint(model)
            self.counter = 0 # Reset bộ đếm về 0
        else:
            # Nếu không cải thiện, tăng bộ đếm thêm 1
            self.counter += 1
            print(f' -> [EarlyStopping] Bộ đếm tăng: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        '''Lưu mô hình khi điểm Validation F1 tăng'''
        torch.save(model.state_dict(), self.path)
        print(f' -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.')

In [11]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

def train_epoch(epoch, EPOCHS):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Train
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Cập nhật số loss liên tục trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    loss_avg = total_loss / len(train_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

@torch.no_grad()
def valid_epoch(epoch, EPOCHS):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Valid
    progress_bar = tqdm(valid_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Valid]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
    loss_avg = total_loss / len(valid_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

In [12]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv

class AdvancedGAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(AdvancedGAT, self).__init__()
        
        # 1. Lớp đầu vào (Giữ nguyên để nén feature)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GAT với Multi-head Attention
        # Lớp 1: Input 32 -> 4 heads x 16 = 64 chiều (Tương đương GCNConv 64)
        self.gat1 = GATConv(
            in_channels=32, out_channels=16, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 2: Input 64 -> 4 heads x 32 = 128 chiều (Tương đương GCNConv 128)
        self.gat2 = GATConv(
            in_channels=64, out_channels=32, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 3: Input 128 -> 4 heads x 64. 
        # CHÚ Ý: Ở lớp GNN cuối cùng, ta dùng concat=False để tính trung bình các heads, 
        # gom lại thành 64 chiều chuẩn bị cho lớp Linear.
        self.gat3 = GATConv(
            in_channels=128, out_channels=64, heads=4, concat=False, edge_dim=1
        )
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # TIỂU XẢO KỸ THUẬT QUAN TRỌNG:
        # GATConv yêu cầu edge_attr phải là ma trận 2 chiều [Số cạnh, Số feature cạnh].
        # Trọng số Cosine + Time Decay của ta đang là mảng 1 chiều [E], ta cần thêm 1 trục (unsqueeze).
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
            
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp GAT 1 (Dùng hàm ELU thay vì ReLU vì các paper về GAT chứng minh ELU mượt mà hơn cho Attention)
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 2
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 3
        x = self.gat3(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [15]:
# Khởi tạo mô hình

# Chuyển mô hình lên GPU (nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Khai báo Optimizer (Adam là chuẩn mực nhất)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# 1. Chạy đoạn code sinh class_weights của bạn
# LƯU Ý: all_train_labels phải là nhãn của TẬP TRAIN (Không gộp Valid/Test vào để tránh Data Leakage)
import sklearn.utils.class_weight as class_weight

unique_classes = np.unique(train_df['label']) # Lấy các class có trong tập train
all_train_labels = train_df['label'].values

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=all_train_labels
)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
class_weights = np.power(class_weights, 0.4) 
class_weights = np.clip(class_weights, a_min=0.5, a_max=6.0) 

# Đưa lên thiết bị (GPU/CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device) 

# ==========================================
# 2. Áp dụng vào Hàm Mất Mát (Loss Function)
# ==========================================
# Chèn class_weights_tensor vào tham số 'weight' của NLLLoss
criterion = torch.nn.NLLLoss(weight=class_weights_tensor)

print("Đã tích hợp Class Weights vào hàm Loss thành công!")
print(f"Trọng số các class: {class_weights}")


Đã tích hợp Class Weights vào hàm Loss thành công!
Trọng số các class: [1.64730877 0.5        3.7653502  1.29446194 1.67705869 3.80442181
 2.0250111  0.83822433 1.76304403 1.22605595 1.69183888]


In [21]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gat_time.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gat_time.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")


AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.3385 - Macro F1: 0.7265 | Valid Loss: 0.1508 - Macro F1: 0.8644
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.1126 - Macro F1: 0.8912 | Valid Loss: 0.1849 - Macro F1: 0.9026
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.0846 - Macro F1: 0.9268 | Valid Loss: 0.1259 - Macro F1: 0.9187
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.0830 - Macro F1: 0.9263 | Valid Loss: 0.1652 - Macro F1: 0.8979
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


...

Epoch: 011/50 | Train Loss: 0.0709 - Macro F1: 0.9429 | Valid Loss: 0.1351 - Macro F1: 0.9167
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 012/50 | Train Loss: 0.0577 - Macro F1: 0.9536 | Valid Loss: 0.2557 - Macro F1: 0.8973
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 013/50 | Train Loss: 0.0530 - Macro F1: 0.9510 | Valid Loss: 0.1639 - Macro F1: 0.8635
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_12368\2559587684.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gat_time.pt

Epoch: 014/50 | Train Loss: 0.0558 - Macro F1: 0.9471 | Valid Loss: 0.1486 - Macro F1: 0.9111
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 14!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!


In [22]:
from sklearn.metrics import classification_report
@torch.no_grad()
def test_benchmark_time_based(loader, model):
    model.eval()
    
    all_preds = []
    all_labels = []
    all_original_indices = [] # Lưu lại index gốc của file CSV
    
    print("Đang chạy Benchmark trên tập Test (Thời gian thực - Delta t = 0.1s)...")
    for batch in tqdm(loader, desc="Testing", ncols=100):
        batch = batch.to(device)
        
        # Forward pass
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Đọc index gốc từ file snapshot truyền qua
        # Do batch gom nhiều đồ thị, batch.ptr sẽ quản lý index nhưng meta thì ta lấy trực tiếp từ batch nếu có
        # Hoặc nếu bạn chạy batch_size=32, hãy đảm bảo cấu trúc data lưu đúng mảng numpy gốc
        if hasattr(batch, 'global_indices'):
            all_original_indices.extend(batch.global_indices)
            
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\n[DONE] Tổng số flow thực tế được đánh giá: {len(all_labels)}")
    print(f"🏆 CHÍNH THỨC - TIME-BASED TEST MACRO F1: {macro_f1:.4f}")
    print("==================================================")
    print(classification_report(all_labels, all_preds, digits=4))
    
    return all_preds, all_labels

In [23]:
model.load_state_dict(torch.load('best_baseline_gat_time.pt'))
model = model.to(device)
test_preds, test_labels = test_benchmark_time_based(test_loader, model)

C:\Users\Admin\AppData\Local\Temp\ipykernel_12368\3762751709.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gat_time.pt'

Đang chạy Benchmark trên tập Test (Thời gian thực - Delta t = 0.1s)...


Testing: 100%|███████████████████████████████████████████████████| 417/417 [00:02<00:00, 142.42it/s]



[DONE] Tổng số flow thực tế được đánh giá: 760240
🏆 CHÍNH THỨC - TIME-BASED TEST MACRO F1: 0.8378
              precision    recall  f1-score   support

           0     0.7208    0.8837    0.7940     19846
           1     0.9997    1.0000    0.9998    484077
           2     0.3245    0.9678    0.4860      2515
           3     0.9961    0.8350    0.9085     36253
           4     0.6950    0.7115    0.7032     18979
           5     0.9551    0.9971    0.9756      2451
           6     0.8291    0.7177    0.7693     11847
           7     1.0000    0.9926    0.9963    107436
           8     0.6470    0.9478    0.7690     16746
           9     0.9997    0.7844    0.8790     41523
          10     0.9553    0.9147    0.9346     18567

    accuracy                         0.9613    760240
   macro avg     0.8293    0.8866    0.8378    760240
weighted avg     0.9708    0.9613    0.9636    760240

